In [58]:
# self.current_positions
ticker_list = ['AMZN', 'TSM', 'NVDA', 'META']



class RandomClass(object):
    def __init__(self, ticker_list):

        self.ticker_list = ticker_list

        self.current_positions = dict( (k,v) for k, v in [(s, 0) for s in self.ticker_list] )
    

        """
        [(S, 0) for s in self.ticker_list]

        - Creates a list of tuples. e.g. [('AAPL', 0), ('TSLA', 0), ...]
        - Each ticker has an initial position of 0

        (k, v) for k, v in [...]
        k = key
        v = value
        
        - Unpacks the list of tuples.

        dict(...)

        - Turns the sequence of (key, value) tuples into a dict

        """


port = RandomClass(ticker_list)
print(port.current_positions)











{'AMZN': 0, 'TSM': 0, 'NVDA': 0, 'META': 0}


In [59]:
import yfinance as yf
import os
ticker_list = ['SPY']
data_dir = '/Users/george/python-projects/ed-backtest/backtester/data/sp_constitutents'

print(ticker_list)
for t in ticker_list:
    ticker = yf.Ticker(f"{t}")
    df = ticker.history(start="2005-01-01", end="2026-01-01")

    filepath = os.path.join(data_dir, f'{t}.csv')
    df.to_csv(filepath)
    print(f"{t} data downloaded to {filepath}")
    

['SPY']


/Users/george/python-projects/ed-backtest/ed_venv/lib/python3.12/site-packages/yfinance/scrapers/history.py:201: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()


SPY data downloaded to /Users/george/python-projects/ed-backtest/backtester/data/sp_constitutents/SPY.csv


In [60]:
"""
get_trade_points() WIN/LOSS TRACKING ORIGINAL LOGIC:


    if not sell_prices[i]:
        continue
    if sell_prices[i] > buy_prices[i]:
        wins += 1
        trade_pnl.append((sell_prices[i] - buy_prices[i]) * pos_sizes[i])
    elif sell_prices[i] < buy_prices[i]:
        losses += 1
        trade_pnl.append((sell_prices[i] - buy_prices[i]) * pos_sizes[i])
    elif sell_prices[i] == buy_prices[i]:
        breakeven += 1
    else:
        continue
"""

'\nget_trade_points() WIN/LOSS TRACKING ORIGINAL LOGIC:\n\n\n    if not sell_prices[i]:\n        continue\n    if sell_prices[i] > buy_prices[i]:\n        wins += 1\n        trade_pnl.append((sell_prices[i] - buy_prices[i]) * pos_sizes[i])\n    elif sell_prices[i] < buy_prices[i]:\n        losses += 1\n        trade_pnl.append((sell_prices[i] - buy_prices[i]) * pos_sizes[i])\n    elif sell_prices[i] == buy_prices[i]:\n        breakeven += 1\n    else:\n        continue\n'

In [61]:
pnl = (186.82 - 183.26) * 136

print(pnl)

484.1600000000003


In [62]:



def f():
    yield 1
    yield 2
    yield 3

g = f()



In [63]:
from queue import Queue

q = Queue()

for i in range(1, 4):
    q.put(f"Event {i}")

print(f"Queue size: {q.qsize()}")
print(f"Is empty? {q.empty()}")

while not q.empty():
    print(q.get())


Queue size: 3
Is empty? False
Event 1
Event 2
Event 3


In [64]:

initial_fill_price = [72.46, 67.85, 152.50, 28.68]

initial_cost = sum(initial_fill_price) * 100

cash = 100000
cash -= initial_cost


final_fill_prices = [190.72, 139.88, 370.51, 248.47]
last_mkt_value = sum(final_fill_prices) * 100

final_portfolio_value = cash + last_mkt_value
print(final_portfolio_value)


# Validating portfolio final value with numbers directly from CSV, correct!


162809.0


In [65]:
from abc import ABC, abstractmethod
from event import SignalEvent

class Strategy(ABC):

    @abstractmethod
    def calc_signals(self):

        raise NotImplementedError("calc_signal required, not implemented.")
    
class SMA_CrossoverStrategy(Strategy):
    def __init__(self, bars, events):

        self.bars = bars
        self.ticker_list = self.bars.ticker_list
        self.events = events
        self.bought, self.short = self._calc_initial_bought()

    def _calc_initial_bought(self):

        bought = {}
        for s in self.ticker_list:
            bought[s] = False

        short = {}
        for s in self.ticker_list:
            short[s] = False

        return bought, short
    

    
    def calc_signals(self, event):
        
        if event.type == "MARKET":
            for s in self.ticker_list:
                bars = self.bars.get_latest_bars(s, N=30)
                
                if bars is not None and bars != []:
                    if len(bars) >= 30:
                        
                        closes = [bar[4] for bar in bars[-30:]]
                        sma = sum(closes) / 30
                        
                        if closes[-1] > sma and self.bought[s] == False:
                            
                            signal = SignalEvent(bars[-1][0], bars[-1][1], 'LONG')
                            self.events.put(signal)
                            self.bought[s] = True

                        elif closes[-1] < sma and self.bought[s] == True:
                            signal = SignalEvent(bars[-1][0], bars[-1][1], 'EXIT')
                            self.events.put(signal)
                            self.bought[s] = False

                        elif closes[-1] < sma and self.bought[s] == False:
                            signal = SignalEvent(bars[-1][0], bars[-1][1], 'SHORT')
                            self.events.put(signal)
                            self.short[s] = True

                        elif closes[-1] > sma and self.short[s] == True:
                            signal = SignalEvent(bars[-1][0], bars[-1][1], 'EXIT')
                            self.events.put(signal)
                            self.short[s] = False
                        
                        else:
                            continue








ModuleNotFoundError: No module named 'event'

In [ ]:
class SMA_CrossoverStrategy(Strategy):
    def __init__(self, bars, events):

        self.bars = bars
        self.ticker_list = self.bars.ticker_list
        self.events = events
        self.bought, self.short = self._calc_initial_bought()

    def _calc_initial_bought(self):

        bought = {}
        for s in self.ticker_list:
            bought[s] = False

        short = {}
        for s in self.ticker_list:
            short[s] = False

        return bought, short
    

    
    def calc_signals(self, event):
        
        if event.type == "MARKET":
            for s in self.ticker_list:
                bars = self.bars.get_latest_bars(s, N=30)
                
                if bars is not None and bars != []:
                    if len(bars) >= 30:
                        
                        closes = [bar[4] for bar in bars[-30:]]
                        sma = sum(closes) / 30
                        
                        if closes[-1] > sma and self.bought[s] == False:
                            print(f"Close: {closes[-1]}")
                            print(f"SMA: {sma}")
                        
                            signal = SignalEvent(bars[-1][0], bars[-1][1], 'LONG')
                            self.events.put(signal)
                            self.bought[s] = True

                        elif closes[-1] < sma and self.bought[s] == True:
                            print(f"Close: {closes[-1]}")
                            print(f"SMA: {sma}")

                            signal = SignalEvent(bars[-1][0], bars[-1][1], 'EXIT')
                            self.events.put(signal)
                            self.bought[s] = False

                        elif closes[-1] < sma and self.bought[s] == False:
                            print(f"Close: {closes[-1]}")
                            print(f"SMA: {sma}")
                            
                            signal = SignalEvent(bars[-1][0], bars[-1][1], 'SHORT')
                            self.events.put(signal)
                            self.short[s] = True

                        elif closes[-1] > sma and self.short[s] == True:
                            print(f"Close: {closes[-1]}")
                            print(f"SMA: {sma}")
                            
                            signal = SignalEvent(bars[-1][0], bars[-1][1], 'EXIT')
                            self.events.put(signal)
                            self.short[s] = False
                        
                        else:
                            continue

In [ ]:
def _open_convert_csv_files(self):
    """Opens the CSV files from the data directory, converts them into pandas DataFrames within a ticker dictionary"""
    """
    The idea is to get the data and merge them into 1 timeline.
    """

    comb_index = None
    for s in self.ticker_list:
        # Load the CSV file with no header information, indexed on date
        #print(s)
        self.ticker_data[s] = pd.read_csv(
            os.path.join(self.csv_dir, '%s.csv' % s),
            header=0, index_col=0, parse_dates=True,
            names=[ # Override the CSV column names and replace with these, for robustness and reusability.
                'datetime', 'open', 'high',
                'low', 'close', 'adj_close', 'volume'
            ]
        )
        self.ticker_data[s].sort_index(inplace=True)
        #print(self.ticker_data[s].close)
        #I've got the whole CSV of data fine.

In [ ]:
import pandas as pd
def _open_convert_csv_files(self):

    comb_index = None
    for s in self.ticker_list:

        self.ticker_data[s] = pd.read_csv(
            os.path.join(self.csv_dir, f'{s}.csv'),
            header=0,
            index_col=0,
            parse_dates=True,
            names=['datetime', 'open', 'high', 'low', 'close', 'adj_close', 'volume']
        )
        self.ticker_data[s].sort_index(inplace=True)


In [ ]:
class DualSMA_CrossoverStrategy(Strategy):
    def __init__(self, bars, events):

        self.bars = bars
        self.ticker_list = self.bars.ticker_list
        self.events = events
        self.bought, self.short = self._calc_initial_bought()
        self.short_period = 50
        self.med_period = 200

    def _calc_initial_bought(self):

        bought = {}
        for s in self.ticker_list:
            bought[s] = False

        short = {}
        for s in self.ticker_list:
            short[s] = False

        return bought, short
    

    
    def calc_signals(self, event):
        
        if event.type == "MARKET":
            for s in self.ticker_list:
                bars = self.bars.get_latest_bars(s, N=self.med_period)
                
                if bars is not None and bars != []:
                    if len(bars) >= self.med_period:
                        closes_med = [bar[4] for bar in bars]
                        closes_short = [bar[4] for bar in bars[-self.short_period:]]
                        sma_med = sum(closes_med) / self.med_period
                        sma_short = sum(closes_short) / self.short_period
                        
                        if sma_short > sma_med and self.bought[s] == False:
                            signal = SignalEvent(bars[-1][0], bars[-1][1], 'LONG')
                            self.events.put(signal)
                            self.bought[s] = True

                        elif sma_short < sma_med and self.bought[s] == True:
                            signal = SignalEvent(bars[-1][0], bars[-1][1], 'EXIT')
                            self.events.put(signal)
                            self.bought[s] = False



In [ ]:
   
"""
MEAN REVERSION STRATEGY
"""


def calc_signals(self, event):
        if event.type == "MARKET":
            bull_regimes = self.is_bull_regime()

            for s in self.ticker_list:
                #if not bull_regimes[s]:
                #    continue

                bars = self.bars.get_latest_bars(s, N=self.z_score_period)
                
                if bars is not None and bars != []:
                    if len(bars) >= self.z_score_period:
                        z_period_closes = np.array([bar[4] for bar in bars])
                        sma = np.mean(z_period_closes)
                        std = np.std(z_period_closes)
                        z_score = (z_period_closes[-1] - sma) / std

                        if z_score < -self.z_condition and not self.bought[s]:
                            print(f"GENERATING BUY SIGNAL FOR {s}: Z_Score = {z_score}")
                            signal = SignalEvent(bars[-1][0], bars[-1][1], 'LONG', use_risk_manager=True)
                            self.events.put(signal)
                            
                            self.bought[s] = True
                            self.short[s] = False
                            self.entry_bar[s] = bars[-1][1]

                        elif z_score > 2 and not self.short[s]:
                            signal = SignalEvent(bars[-1][0], bars[-1][1], 'SHORT', use_risk_manager=True)
                            self.events.put(signal)
                            self.short[s] = True
                            self.bought[s] = False
                            self.entry_bar[s] = bars[-1][1]


                        elif abs(z_score) < 0.5:
                            current_bar = bars[-1][1]

                            if self.bought[s] and current_bar > self.entry_bar[s]:
                                signal = SignalEvent(bars[-1][0], bars[-1][1], 'EXIT')
                                self.events.put(signal)
                                self.bought[s] = False
                                self.entry_bar[s] = None
                                
                            if self.short[s]:
                                signal = SignalEvent(bars[-1][0], bars[-1][1], 'EXIT')
                                self.events.put(signal)
                                self.short[s] = False
                                self.entry_bar[s] = None


In [ ]:

digits = [9, 9, 9]


value = 0
mult_factor = 1
digits = digits[::-1]
new_digits = []

for digit in digits:
    value += digit * mult_factor
    mult_factor = mult_factor * 10

value += 1

length = len(str(value))
for i in range(length):
    div_factor = 1 * (10 ** (length - 1))
    new_digits.append(value // div_factor)
    value -= (value // div_factor * (10 ** (length-1)))
    length -= 1

print(new_digits)






[1, 0, 0, 0]


In [ ]:


def find_index_of_first_occurence(haystack, needle):
    for i in range(len(haystack) - len(needle) + 1):
        value = haystack[i: i + len(needle)]
        if value == needle:
            return i
    return -1

print(find_index_of_first_occurence(haystack="sasadbustusad", needle="sad"))


# haystack[i : i + len(needle)] : E.g. haystack[0:2] gives all values from index [0] up to and NOT including index [2].

# so it will return [0] and [1], ONLY





2


In [ ]:

def length_of_last_word(s):
    s = s.split()
    return len(s[-1])


print(length_of_last_word(s="Hello World"))


def harder_length_of_last_word(s):
    count = 0
    i = len(s) - 1

    while i >= 0 and s[i] == " ":
        i-=1
    while i >= 0 and s[i] != " ":
        count += 1
        i-= 1
    return count

print(harder_length_of_last_word(s="Hello Worlds   "))







5
6


In [ ]:
words = "HELLO WORLD BOY   "

count = 0
i = len(words) - 1

while i >= 0 and words[i] == " ":
    i -= 1

while i >= 0 and words[i] != " ":
    count += 1
    i -= 1


print(count)



3


In [ ]:
haystack = "sadbastiosad"
needle = "sad"

for i in range(len(haystack) - len(needle) + 1):
    value = haystack[i: i + len(needle)]
    if value == needle:
        print(i)
        break
    else:
        print(-1)
        break


0


In [ ]:
# My Soltuion to Length of Longest Substring

def LengthOfLongestSubstring(s: str) -> int:
    seen = []
    count = 0
    highest_count = 0

    for i in s:
        if i not in seen:
            seen.append(i)
            count = len(seen)
            if count > highest_count:
                highest_count = count
        elif i in seen:
            pos = seen.index(i)
            del seen[0: pos+1]
            seen.append(i)
            count = len(seen)
            if count > highest_count:
                highest_count = count
    return highest_count


print(LengthOfLongestSubstring(s="bbbbbasd"))


# Each of the following commands make this solution O(n) time complexity:
#   - seen.index(i)
#   - if in seen
#   - if not in seen

# Stacked together, the worst case is that this solution is O(n**2)


# For O(1) time complexity, the soltion requires a hash map, to directly search for a value







4


In [ ]:
def RLE(n: int) -> str:
    rle = "1"
    for i in range(n - 1):
        term_string = ""
        count = 1
        prev = rle[0]
        for j in rle[1:]:

            if j == prev:
                count += 1
            else:
                term_string += str(count) + prev
                count = 1
                prev = j


        term_string += str(count) + prev
        rle = term_string


    return rle

print(RLE(n=4))


KeyboardInterrupt: 

In [ ]:
ticker_list = []





Should sell GTA.
Should sell SME.


In [ ]:
from queue import Queue

pending_orders = Queue()

print(f"Initial Queue size: {pending_orders.qsize()}")

pending_orders.put("MSFT")
pending_orders.put("TSLA")
pending_orders.put("AAPL")

while True:
    order = pending_orders.get()
    print(order)
    print(pending_orders.qsize())







In [ ]:
import numpy as np

price = 194.27
quantity = 1000
direction = "BUY/SELL"

slippage = round(price * 0.0001 * np.log10(2*float(quantity)), 2)

if direction == "BUY":
    fill_price = price + slippage
else:
    fill_price = price - slippage


print(fill_price)


0.06
194.21


In [ ]:
import pandas as pd
values1 = [1.0, 1.0, 1.0, 1.1, 1.2]
values2 = [1.0, 1.3, 1.2, 1.2, 1.4]
values3 = [1.0, 1.2, 1.1, 1.3, 1.4]

values1 = pd.Series(values1)
values2 = pd.Series(values2)
values3 = pd.Series(values3)

all_values = []
all_values.append(values1)
all_values.append(values3)
all_values.append(values2)

concat_values = []
multiplier = float(0)
for i in range(len(all_values)):

    if i == 0:
        concat_values.append(all_values[i])
        multiplier = all_values[i].iloc[-1]
        continue
    else:
        temp = all_values[i] * multiplier
        concat_values.append(temp)
        multiplier = temp.iloc[-1]


full_curve = pd.concat(concat_values)
print(full_curve)




0    1.000
1    1.000
2    1.000
3    1.100
4    1.200
0    1.200
1    1.440
2    1.320
3    1.560
4    1.680
0    1.680
1    2.184
2    2.016
3    2.016
4    2.352
dtype: float64


In [ ]:
import sqlite3


con = sqlite3.connect("attempt1.db")
cur = con.cursor()

#cur.execute("SELECT name FROM sqlite_master WHERE type='table';")
#print(cur.fetchall())

# BUILD BETTER REGIME CLASSIFIER, AND PARAMETER SWITCH BETWEEN REGIMES



[('results',)]


In [ ]:
# SQL DATABASE ACCESS

con = sqlite3.connect("attempt1.db")
cur = con.cursor()


cur.execute("""
INSERT INTO results (date, sharpe, lookback, reshuffle, topn)
VALUES ('2020-2022', 1.2, 252, 21, 5)
""")

con.commit()

In [ ]:
res = cur.execute("SELECT * FROM results")
print(res.fetchall())

[('2020-2022', 1.2, 252, 21, 5)]


In [ ]:
# VOLATILITY CALCULATIONS

import numpy as np
import pandas as pd
import os

data_dir = "/Users/george/python-projects/ed-backtest/backtester/data/sp_constitutents"
filepath = os.path.join(data_dir, "AAPL.csv")
data = pd.read_csv(filepath)

intraday_vol = None
lookback = 30

returns = data['Close'][-lookback:].pct_change().dropna()
#returns = closes.pct_change().dropna()
print(returns)

current_vol = returns.std()


"""
Should I calculate volatility before receiving the data, or does that have to happen in strategy?

I think strategy should do this work.

Aha, i've got it, use the rolling window volatility to get an idea of vol within the window. Then use current vol calculation to 
decide whether to enter a position.


"""
print(f"Current Volatility: {current_vol}")
#print(data)



In [ ]:
# POTENTIAL FEATURE ENGINE CLASS
class FeatureEngine():
    """
    Every "heartbeat" in the system, feature_engine.update() is called to get the latest market_data()


    """
    def __init_(self, features):
        self.features = features
        self.feature_values = {}
        







            





In [ ]:
# POTENTIAL FEATURE CLASSES
from abc import ABC, abstractmethod
import pandas as pd
import numpy as np

class Feature(ABC):
    """
    Abstract base class for any Feature.
    
    """
    
    def __init__(self, name):
        self.name = name
        
        pass


    @abstractmethod
    def calc_feature(self, bars):

        raise NotImplementedError("Should add calc_feature func.")




class VolFeature(Feature):
    """
    Volatility Feature which calculates std(returns) based on a given lookback period.

    By passing a lookback value at instantiation, the object is easily implementable into grid-searching
    methods.

    """


    def __init__(self, lookback):
        self.name = "volatility"
        self.lookback = lookback


    def calc_feature(self, data):
        returns = data[-self.lookback:].pct_change().dropna()
        return returns.std()
    


    


In [ ]:
# REFRESHING FUNCTION: UPDATE_TIMEINDEX()

# STEP 1: EVENT SEEDING
from queue import Queue

class MarketEvent():
    def __init__(self, datetime):
        self.type = 'MARKET'
        self.datetime = datetime

def stimulate_data_arrival(events):

    event = MarketEvent("2020-01-01-04:00:00")
    events.put(event)


# STEP 2: REFRESH

def update_timeindex(self, event):
    
    bars = {}
    for ticker in self.ticker_list:
        bars[ticker] = self.bars.get_latest_bars(ticker, N=1)


    positions_snapshot = dict.fromkeys(self.ticker_list, 0)
    positions_snapshot['datetime'] = bars[self.ticker_list[0]][0][1]

    for ticker in self.ticker_list:
        positions_snapshot[ticker] = self.current_positions[ticker]

    self.all_positions.append(positions_snapshot)


    holdings_snapshot = dict.fromkeys(self.ticker_list, 0)
    holdings_snapshot['datetime'] = bars[self.ticker_list[0]][0][1]
    holdings_snapshot['cash'] = self.current_holdings['cash']
    holdings_snapshot['commission'] = self.current_holdings['commission']
    holdings_snapshot['slippage'] = self.current_holdings['slippage']
    holdings_snapshot['total'] = self.current_holdings['cash']

    for ticker in self.ticker_list:
        bars = bars[ticker]

        if bars is None or len(bars) == 0:
            continue

        market_value = self.current_positions[ticker] * bars[0][4]
        holdings_snapshot[ticker] = market_value
        holdings_snapshot['total'] += market_value
    
    self.current_holdings['total'] = holdings_snapshot['total']
    self.all_holdings.append(holdings_snapshot)





# STEP 1: EVENT SEEDING

events = Queue()
stimulate_data_arrival(events)



while True:
    if events.empty():
        break

    event = events.get()
    if event.type == "MARKET":
        update_timeindex(event)







Market event received.


In [ ]:

ticker_list = ['AAPL', 'MSFT', 'TSLA']

positions_snap = dict((k, v) for k, v in [(s, 0) for s in ticker_list])

print(positions_snap)

pos_snapshot = dict.fromkeys(ticker_list, 0)
pos_snapshot['datetime']

print(pos_snapshot)

{'AAPL': 0, 'MSFT': 0, 'TSLA': 0}
{'AAPL': 0, 'MSFT': 0, 'TSLA': 0}


In [ ]:
# ORIGINAL TIMEINDEX FUNC


bars = {}
for ticker in self.ticker_list:
    bars[ticker] = self.bars.get_latest_bars(ticker, N=1)

# Update positions
dp = dict( (k,v) for k, v in [(s, 0) for s in self.ticker_list] )
dp['datetime'] = bars[self.ticker_list[0]][0][1]

for s in self.ticker_list:
    dp[s] = self.current_positions[s]

self.all_positions.append(dp)

dh = dict( (k,v) for k, v in [(s, 0) for s in self.ticker_list] )
dh['datetime'] = bars[self.ticker_list[0]][0][1]
dh['cash'] = self.current_holdings['cash']
dh['commission'] = self.current_holdings['commission']
dh['slippage'] = self.current_holdings['slippage']
dh['total'] = self.current_holdings['cash']

for s in self.ticker_list:
    bars_s = bars[s]

    if bars_s is None or len(bars_s) == 0:
        continue

    market_value = self.current_positions[s] * bars_s[0][4]
    dh[s] = market_value
    dh['total'] += market_value

self.current_holdings['total'] = dh['total']
self.all_holdings.append(dh)